In [1]:
import numpy as np 
import pandas as pd 
import re
from nltk.corpus import stopwords
import nltk
from nltk.stem import PorterStemmer
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/ammar-y530/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [2]:
data_path = r"/home/ammar-y530/Documents/S2/SEM 2/RFPP/RFPP_Task/Task N-grams/abstract.csv"
df = pd.read_csv(data_path, sep=";")
df = df.drop(columns=['abstrak_idn'])
df.head()

,title,abstrak_eng
0,Transfer Learning of Pre-trained Transformers ...,"Nowadays, internet has become the most popular..."
1,Personality Classification of Myers Briggs Typ...,Personality classification using textual data ...
2,Developments and Trends in Indonesian Tourism ...,"Information technology has changed society, se..."
3,Offensive Language and Hate Speech Detection U...,Hate speech detection is a crucial issue...
4,IndoBERT Optimization for Sentiment Analysis o...,"In the digital era, sentiment analysis ha..."


In [3]:
class TextPreprocessor(BaseEstimator, TransformerMixin):
    def __init__(self, stop_words=None):
        self.stop_words = stop_words if stop_words else set()
        self.stemmer = PorterStemmer()

    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        cleaned_texts = []
        for text in X:
            text = text.lower()
            text = re.sub(r'\d+', '', text)
            text = re.sub(r'[^\w\s]', '', text)
            text = re.sub(r'\s+', ' ', text).strip()
            tokens = text.split()
            # tokens = [word for word in tokens if word not in self.stop_words]
            tokens = [self.stemmer.stem(word) for word in tokens]
            # cleaned_texts.append(tokens)
            cleaned_texts.append(" ".join(tokens))
        return cleaned_texts

class TrigramExtractor(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        trigrams_docs = []
        for tokens in X:
            trigrams = [
                " ".join(tokens[i:i+3])
                for i in range(len(tokens)-2)
            ]
            trigrams_docs.append(trigrams)
        return trigrams_docs

stop_words = set(stopwords.words('english'))

In [4]:
pipeline = Pipeline([
    ("cleaning", TextPreprocessor(stop_words=stop_words)),
    # ("trigram", TrigramExtractor())
])

In [5]:
preprocessed_output = pipeline.fit_transform(df['abstrak_eng'])
print(preprocessed_output)

['nowaday internet ha becom the most popular sourc of news howev the valid of the onlin news articl is difficult to assess whether it is a fact or a hoax hoax relatedto covidbrought a problemat effect to human life an accur hoax detect system is import to filter abund inform on the internet in thi researcha covid hoax detectionsystemwaspropos by transfer learn of pretrainedtransform model finetun origin pretrain bert multilingu pretrain mbert and monolingu pretrain indobertwer usedto solv the classif taskin the hoax detect system base on the experiment result finetun indobert model train on monolingu indonesian corpu outperform finetun origin and multilingu bert with uncas version howev the finetun mbert case model trainedona larger corpu achiev the best perform', 'person classif use textual data from social media or onlin forum is a complex task due to the unstructur text and the multifacet natur of person while the myersbrigg type indic mbti provid a comprehens framework adapt it to 

In [6]:
# vectorizer = TfidfVectorizer(ngram_range=(3, 3)) # Pembuatan 3-gram
# tfidf_matrix = vectorizer.fit_transform(preprocessed_output)
# print(tfidf_matrix.shape)

vectorizer = CountVectorizer(ngram_range=(3, 3))  # 3-gram tanpa TF-IDF
count_matrix = vectorizer.fit_transform(preprocessed_output)
print(count_matrix.shape)

(5, 798)


In [9]:
query = "In  the  digital  era,  sentiment  analysis has  become  crucial  to  evaluate  public  opinion, especially in the context of Play Storeapps with Indonesian-language reviews. This research aims to  improve  the  performance  of  the  IndoBERT  model  in  sentiment  analysis  of  DeepSeek  app reviews by using data augmentation and hyperparameter tuning techniques. Data augmentation is  done  through the back-translation  technique,  while  the  hyperparameters  tested  include  the number of epochs, learning rate, and batch size. Experimental results show that the combination of data augmentation with epoch 10, learning rate 2e-5, and batch size 16 produces thehighest accuracy  of  93.95%  and  F1-score  of  0.94,  with  better  stability  than  the  model  without augmentation. The model without augmentation showed fluctuations in performance, indicating overfitting   in   some   configurations.   These   findings   confirm   the   importance   of   applying augmentation techniques and hyperparameter tuning in improving the accuracy and stability of sentiment analysis models, and contribute to the development of NLP models for Indonesian and other resource-constrained languages."
# query = "BERT, hoax, detection"

# stemmed_q = [PorterStemmer().stem(word) for word in query.split()]
# stemmed_q = " ".join(stemmed_q)
# query_vector = vectorizer.transform([stemmed_q])
# scores = cosine_similarity(query_vector, tfidf_matrix).flatten()

stemmed_q = [PorterStemmer().stem(word) for word in query.split()]
stemmed_q = " ".join(stemmed_q)
query_vector = vectorizer.transform([stemmed_q])
scores = cosine_similarity(query_vector, count_matrix).flatten()

In [10]:
print(scores)

[0.         0.03315988 0.03274485 0.01931348 0.86541449]
